# Sparse rewards and hindsight goals for UR5e pick-and-place

**Dev Kumar · Robotics / Reinforcement Learning · CPU only**

This tutorial isolates the sparse goal-distance reward used in my [UR5e pick-and-place environment](https://github.com/devkumar-projects/ur5e-ReinforcementLearning-pick-and-place/blob/main/ur5e_rl_gazebo/env.py), and illustrates a single hindsight goal relabelling with a synthetic trajectory.

It does **not** train an agent or run Gazebo, use a physical robot, or measure success rates. The numerical positions below are deliberately illustrative; they are not experimental robot data.

## 1. The project’s sparse reward

The original `compute_reward` returns **0 when the Euclidean distance is at most 0.05 m, and −1 when it is greater**. At exactly 0.05 m the source uses `>` rather than `>=`. We keep that boundary here. For this small NumPy example, the function accepts both individual and batched goal arrays.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SUCCESS_THRESH = 0.05  # metres, from the source environment

def sparse_reward(achieved_goal, desired_goal):
    distance = np.linalg.norm(np.asarray(achieved_goal) - np.asarray(desired_goal), axis=-1)
    return -(distance > SUCCESS_THRESH).astype(np.float32)

origin = np.array([0., 0., 0.])
test_goals = np.array([[0., 0., 0.], [0.04, 0., 0.], [0.05, 0., 0.], [0.06, 0., 0.]])
rewards = sparse_reward(test_goals, origin)
assert np.array_equal(rewards, [0., 0., 0., -1.])
assert sparse_reward(test_goals[1], origin) == 0.
display(pd.DataFrame({"distance_m": np.linalg.norm(test_goals, axis=1), "reward": rewards}))

## 2. Relabel a goal using a later achieved position

HER can reuse an observed transition under a goal that was actually achieved later in the same episode. Here we choose the final position as a **single illustrative hindsight goal**, keep the achieved positions unchanged, and recalculate the sparse reward. This is not a complete HER replay implementation.

In [ ]:
# Synthetic 3D positions in metres, not Gazebo measurements.
steps = np.arange(7)
achieved = np.array([[0., 0., 0.], [0.03, 0.01, 0.], [0.06, 0.02, 0.],
                     [0.09, 0.035, 0.], [0.12, 0.05, 0.], [0.15, 0.06, 0.],
                     [0.18, 0.07, 0.]])
original_goal = np.array([0.35, 0.10, 0.0])
hindsight_goal = achieved[-1].copy()  # a later state in this same synthetic episode
r_original = sparse_reward(achieved, original_goal)
r_hindsight = sparse_reward(achieved, hindsight_goal)
assert r_original.shape == r_hindsight.shape == (len(steps),)
assert r_original[-1] == -1 and r_hindsight[-1] == 0
comparison = pd.DataFrame({"step": steps,
    "distance_original_m": np.linalg.norm(achieved-original_goal, axis=1),
    "reward_original": r_original,
    "distance_hindsight_m": np.linalg.norm(achieved-hindsight_goal, axis=1),
    "reward_hindsight": r_hindsight})
display(comparison.round(3))

In [ ]:
fig, (ax, reward_ax) = plt.subplots(1, 2, figsize=(12, 4))
ax.plot(achieved[:,0], achieved[:,1], "o-", color="#117e83", label="Achieved positions")
ax.scatter(*original_goal[:2], marker="*", s=180, color="#cf6a30", label="Original goal")
ax.scatter(*hindsight_goal[:2], marker="X", s=100, color="#514b9e", label="Hindsight goal")
ax.set(xlabel="x (m)", ylabel="y (m)", title="Illustrative trajectory (z = 0)")
ax.axis("equal"); ax.grid(alpha=.2); ax.legend()
reward_ax.step(steps, r_original, where="mid", label="Original goal", color="#cf6a30")
reward_ax.step(steps, r_hindsight, where="mid", label="Hindsight goal", color="#514b9e")
reward_ax.set(yticks=[-1, 0], ylim=(-1.2, .2), xlabel="Step", ylabel="Sparse reward",
              title="Reward under two goals")
reward_ax.grid(alpha=.2); reward_ax.legend(); fig.tight_layout(); plt.show()

## 3. What this illustration does—and does not—show

Relabelling changes the goal and reward assigned to an existing transition; it does not change the recorded state transition or retroactively prove that the original goal was reached. The 5 cm threshold and reward definition are specific to the source environment.

A practical HER implementation must ensure that relabelled transitions are valid for the environment and handle episode termination, truncation and replay consistently. The original project uses a goal-conditioned environment; consult its training and environment code for the actual implementation. Compare independently seeded training and held-out evaluation before making performance claims.

### Source and licence

Reward adapted from [UR5e Reinforcement Learning Pick-and-Place](https://github.com/devkumar-projects/ur5e-ReinforcementLearning-pick-and-place/blob/main/ur5e_rl_gazebo/env.py), MIT licence, copyright © 2026 Dev Kumar. The [full MIT notice](https://github.com/devkumar-projects/ur5e-ReinforcementLearning-pick-and-place/blob/main/LICENSE) remains applicable to the original code. The Kaggle version of this notebook is distributed under Apache 2.0.